# Contrastive RL (Acme-free port) — Colab runner

Binary **NCE** (contrastive_nce) on continuous-control goal-conditioned tasks.
All logic lives in the `crl/` package; this notebook only wires up Colab.

**How to use:** run cells 1-4 once, set `ENV_NAME` / `SEED` in **cell 5**, then
run cells 6-8. Checkpoints + metrics save to Google Drive on every eval.

No dataset needed (online RL). Time on a T4: `fetch_reach` ~20-40 min (300k),
`fetch_push` ~1 h (500k). Watch the printed `sps=`; ETA = steps / sps / 60 min.

In [1]:
# 1. Install dependencies (~2 min).
!pip -q install "jax[cuda12]" dm-haiku optax gymnasium gymnasium-robotics mujoco imageio
import os
os.environ["MUJOCO_GL"] = "egl"  # headless MuJoCo rendering backend
import jax; print("JAX devices:", jax.devices())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.7/232.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.3/374.3 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.2/26.2 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.5/805.5 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 21.3 MB/s eta 0:00:00
JAX devices: [CudaDevice(id=0)]


In [2]:
# 2. Get / update the code (your fork).
import os
os.chdir("/content")
if not os.path.exists("/content/contrastive_rl"):
    !git clone https://github.com/tingrui-huang/contrastive_rl.git
%cd /content/contrastive_rl
!git pull

Cloning into 'contrastive_rl'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 75 (delta 21), reused 57 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (75/75), 366.38 KiB | 2.05 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/contrastive_rl
Already up to date.


In [3]:
# 3. Mount Google Drive (checkpoints + reports are written here).
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
# 4. ===== SET THESE, then run the rest =====
ENV_NAME = "fetch_push"      # "fetch_reach" or "fetch_push"
SEED     = 0                 # 0, 1, 2, ... (one run per seed)
STEPS    = 500_000           # fetch_reach: 300_000 ;  fetch_push: 500_000+
RESUME   = False             # True to continue a previous run of same ENV/SEED
# ==========================================
RUN_DIR = f"/content/drive/MyDrive/contrastive_rl_runs/{ENV_NAME}_nce_s{SEED}"
REPORT_DIR = f"/content/drive/MyDrive/repro_report_{ENV_NAME}_nce"
print("ENV_NAME:", ENV_NAME, "| SEED:", SEED, "| STEPS:", STEPS, "| RESUME:", RESUME)
print("RUN_DIR :", RUN_DIR)

ENV_NAME: fetch_push | SEED: 0 | STEPS: 500000 | RESUME: False
RUN_DIR : /content/drive/MyDrive/contrastive_rl_runs/fetch_push_nce_s0


In [ ]:
# 5. Train the binary-NCE baseline (guard-railed: refuses non-NCE configs).
from crl.config import Config
from crl.train import train

config = Config(
    env_name=ENV_NAME,
    use_td=False, twin_q=False,       # binary NCE (contrastive_nce)
    entropy_coefficient=0.0,
    max_number_of_steps=STEPS,
    min_replay_size=10_000, random_steps=10_000,
    num_sgd_steps_per_step=4,
    eval_every_steps=10_000, eval_episodes=20,
    seed=SEED,
    ckpt_dir=RUN_DIR,
    resume=RESUME,
)
assert config.use_td is False and config.twin_q is False, "must be binary NCE"
print(f"OK: {ENV_NAME}, binary NCE, seed {SEED} -> {RUN_DIR}")
state = train(config)

OK: fetch_push, binary NCE, seed 0 -> /content/drive/MyDrive/contrastive_rl_runs/fetch_push_nce_s0
Config: Config(env_name='fetch_push', max_number_of_steps=500000, obs_dim=-1, goal_dim=-1, action_dim=-1, max_episode_steps=-1, start_index=0, end_index=-1, batch_size=256, actor_learning_rate=0.0003, learning_rate=0.0003, discount=0.99, entropy_coefficient=0.0, target_entropy=0.0, tau=0.005, hidden_layer_sizes=(256, 256), repr_dim=64, repr_norm=False, repr_norm_temp=True, use_cpc=False, use_td=False, add_mc_to_td=False, use_gcbc=False, twin_q=False, random_goals=0.5, use_image_obs=False, min_replay_size=10000, max_replay_size=1000000, updates_per_step=1, num_sgd_steps_per_step=4, random_steps=10000, jit=True, seed=0, eval_every_steps=10000, eval_episodes=20, log_every_steps=1000, ckpt_dir='/content/drive/MyDrive/contrastive_rl_runs/fetch_push_nce_s0', ckpt_every_steps=0, resume=False)


AdroitHandRelocateDense-v1, AdroitHandHammerDense-v1, AdroitHandDoorDense-v1 environment's reward functions were updated in v1.2.1 without an environment version update. Therefore, use gymnasium-robotics==1.2.0 for v1 reproducibility or use v2 in gymnasium-robotics>=1.4.3. See https://github.com/Farama-Foundation/Gymnasium-Robotics/pull/220 for more details


obs_dim=25 goal_dim=3 action_dim=4 max_episode_steps=50 goal_slice=[3:6]
[step     1000] sps=   484 (filling buffer 1000/10000)
[step     2000] sps=   484 (filling buffer 2000/10000)
[step     3000] sps=   487 (filling buffer 3000/10000)
[step     4000] sps=   430 (filling buffer 4000/10000)
[step     5000] sps=   395 (filling buffer 5000/10000)
[step     6000] sps=   408 (filling buffer 6000/10000)
[step     7000] sps=   419 (filling buffer 7000/10000)
[step     8000] sps=   425 (filling buffer 8000/10000)
[step     9000] sps=   432 (filling buffer 9000/10000)
[step    10000] sps=   351 critic=0.032 actor=7.174 cat_acc=0.004
  >> EVAL step 10000: success_rate=0.100 final_dist=0.169 min_dist=0.169
    [ckpt] new best success=0.100 -> best.pkl
    [ckpt] saved step 10000 -> /content/drive/MyDrive/contrastive_rl_runs/fetch_push_nce_s0/latest.pkl
[step    11000] sps=   265 critic=0.023 actor=5.286 cat_acc=0.016
[step    12000] sps=   253 critic=0.020 actor=7.943 cat_acc=0.030
[step    130

In [ ]:
# 6. Build the reproduction report (plots + GIFs + csv + audit) for this run.
!python -m crl.repro_report --env_name {ENV_NAME} --batch_size 256 --run_dirs {RUN_DIR} --out {REPORT_DIR}

In [ ]:
# 7. Show the report figures + rollout GIFs inline.
import os
from IPython.display import Image, display
for f in ["learning_curves.png", "distance_curves.png", "contrastive_logits.png",
          "ranking_accuracy.png", "rollout_random.gif", "rollout_trained.gif"]:
    p = os.path.join(REPORT_DIR, f)
    if os.path.exists(p):
        print(f); display(Image(p))

## Multi-seed report (optional)
For mean±std, run cells 4-6 with `SEED = 0`, then `1`, then `2` (three separate
runs), then run the cell below to combine them into one report.

In [ ]:
# 8. (optional) Combine seeds 0/1/2 into one mean+-std report.
ENV = "fetch_reach"   # or "fetch_push"
base = "/content/drive/MyDrive/contrastive_rl_runs"
d0, d1, d2 = (f"{base}/{ENV}_nce_s0", f"{base}/{ENV}_nce_s1", f"{base}/{ENV}_nce_s2")
out = f"/content/drive/MyDrive/repro_report_{ENV}_nce_3seed"
!python -m crl.repro_report --env_name {ENV} --batch_size 256 --run_dirs {d0} {d1} {d2} --out {out}

## Notes
- **fetch_push is hard:** watch `final_dist` / `min_dist` (object -> target) — they
  fall before success rises. If `final_dist` is still dropping at 500k, raise `STEPS`.
- **Disconnected?** Re-run cells 1-4 with the same `SEED`, set `RESUME = True`, and
  run cell 5 to continue from `latest.pkl`.
- **Checkpoints:** `latest.pkl` (resume), `best.pkl` (best eval), `metrics.json`.
- Existing FetchReach seed-0 run may be named `fetch_reach_nce`; the multi-seed
  cell expects `fetch_reach_nce_s0` — rerun seed 0 with this notebook, or edit `d0`.